# 실습 02 · Iris 표형 데이터 MLP 다중분류 · QUIZ

이 노트북은 tutorial을 완료한 후 직접 실습하는 것입니다.
**문제만 주어집니다.** 각 문제를 풀고 정답과 비교하세요.

**시간:** 약 30분  
**평가:** 데이터 누수 방지, shape 이해, 학습 곡선 해석

**⭐ 푸는 방법:** 각 코드 셀에 뼈대 코드가 주어집니다. `____` 부분만 채운 뒤 실행하세요.
막히면 tutorial의 해당 섹션을 참고해도 됩니다 — 그것도 정식 방법입니다.

## 환경 설정 및 데이터 로드

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("data 폴더를 찾지 못했습니다. 노트북과 같은 위치(또는 상위)에 data 폴더가 있어야 합니다.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
IRIS_PATH = DATA_ROOT / "Iris.csv"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

iris = pd.read_csv(IRIS_PATH)
print("data shape:", iris.shape)
display(iris.head())

## 문제 1: feature와 label 분리, class mapping

iris 데이터에서:
1. feature column은 무엇인가요? (`Id`와 `Species` 제외)
2. label column `Species`를 정수 인덱스로 변환하세요.
3. `X` (150, 4)와 `y` (150,) shape를 확인하세요.
4. class mapping dictionary를 출력하세요.

**예상:**
- feature: SepalLengthCm, SepalWidthCm, PetalLengthCm, PetalWidthCm
- class mapping: {'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
feature_cols = [c for c in iris.columns if c not in {"Id", "____"}]   # 정답 열도 제외
class_names = sorted(iris["Species"].unique())
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

X = iris[____].to_numpy(dtype=np.float32)                # 입력 항목 열들만
y = iris["Species"].map(____).to_numpy(dtype=np.int64)   # 이름 → 번호 대응표 적용

print("X:", X.shape, "y:", y.shape)   # (150, 4), (150,)
print("class mapping:", class_to_idx)

## 문제 2: train/validation/test split

다음을 수행하세요:
1. 전체 데이터 (150개)를 40% test, 60% train으로 분할하세요.
2. train (60%)을 다시 20% validation으로 분할하세요. (train: 48%, val: 12%, test: 40%)
3. `train_test_split`에서 `stratify=y`를 사용하여 class 분포를 유지하세요.
4. 각 split의 크기와 class 분포를 출력하세요.

**예상:** train/val/test ≈ 90/30/30 (또는 비슷한 비율)

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
indices = np.arange(len(y))
train_idx, temp_idx = train_test_split(indices, test_size=____, stratify=____, random_state=SEED)   # 40%를 떼고, 품종 비율 유지
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=y[temp_idx], random_state=SEED)

print("split sizes:", len(train_idx), len(val_idx), len(test_idx))   # 90 30 30
print("train class count:", np.bincount(y[train_idx]))

## 문제 3: train-only 표준화 (누수 방지)

다음을 수행하세요:
1. `StandardScaler().fit(X[train_idx])`로 train 통계량만으로 scaler를 학습하세요.
2. scaler를 사용하여 X_train, X_val, X_test를 각각 변환하세요.
3. train의 평균이 ~0, 표준편차가 ~1인지 확인하세요.
4. val/test의 평균이 train과 다른지 확인하세요 (이는 정상입니다).

**왜:** validation이나 test 통계량을 사용하면 "데이터 누수"가 발생합니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
scaler = StandardScaler().fit(X[____])   # 어느 그룹만 보고 기준을 만들어야 할까요?
X_train = scaler.transform(X[train_idx]).astype(np.float32)
X_val = scaler.transform(X[____]).astype(np.float32)
X_test = scaler.____(X[test_idx]).astype(np.float32)   # 기준 '적용'에 해당하는 동작
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

print("train mean:", np.round(X_train.mean(axis=0), 3))   # 전부 0 근처여야 합니다
print("val mean:", np.round(X_val.mean(axis=0), 3))       # 0이 아니어도 정상입니다

## 문제 4: DataLoader 구성

다음을 수행하세요:
1. train, val, test를 모두 DataLoader로 변환하세요 (batch_size=16).
2. train loader는 `shuffle=True`, val/test는 `shuffle=False`로 설정하세요.
3. 첫 번째 batch의 shape를 출력하세요.
4. label dtype이 `long`인지 확인하세요.

**예상:** input (16, 4), label (16,) with dtype=int64

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
def make_loader(X_array, y_array, batch_size, shuffle):
    dataset = TensorDataset(torch.from_numpy(X_array), torch.from_numpy(y_array))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)

BATCH_SIZE = 16
train_loader = make_loader(X_train, ____, BATCH_SIZE, ____)   # train의 정답 / 학습용은 섞을까요?
val_loader = make_loader(X_val, y_val, BATCH_SIZE, False)
test_loader = make_loader(X_test, y_test, BATCH_SIZE, ____)

xb, yb = next(iter(train_loader))
print("input:", xb.shape, xb.dtype)
print("label:", yb.shape, yb.dtype)   # int64(long)인지 확인

## 문제 5: MLP 모델 정의

다음을 수행하세요:
1. `nn.Sequential`로 다음 구조를 만드세요:
   - Linear(4, 16)
   - ReLU()
   - Linear(16, 3)
2. 모델을 DEVICE로 옮기세요.
3. 샘플 input (2, 4)를 통과시켜 output shape를 확인하세요.
4. 파라미터 총 개수를 계산하세요.

**예상:**
- output shape: (2, 3)
- parameters: 4*16 + 16 + 16*3 + 3 = 131

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
model = nn.Sequential(
    nn.Linear(____, 16),   # 입력 항목 수
    nn.____(),             # 중간에 끼우는 걸러내기 장치
    nn.Linear(16, ____),   # 품종 수
).to(DEVICE)

with torch.no_grad():
    sample_logits = model(xb.to(DEVICE))
print("input → logits:", xb.shape, "→", sample_logits.shape)          # (16, 3)
print("parameters:", sum(p.numel() for p in model.parameters()))     # 131이어야 합니다

## 문제 6: 학습·검증 루프 함수 구현

다음 함수를 구현하세요:

```python
def run_epoch(model, loader, criterion, optimizer=None):
    # optimizer가 있으면 학습, 없으면 검증
    # 반환: (평균 loss, 정확도)
```

기능:
1. `optimizer is not None`으로 training 모드 판단
2. 각 batch에 대해 forward → loss 계산
3. training일 때만 backward와 optimizer.step()
4. 전체 loss와 accuracy를 누적하여 반환

**힌트:** `logits.argmax(1) == yb`로 예측값 비교

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not ____        # optimizer를 받았으면 학습 모드
    model.train(____)                       # 모델에 현재 모드를 알림
    total_loss = total_correct = total_count = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                loss.____()                 # 수정 방향 계산
                optimizer.____()            # 수정 실행
        total_loss += loss.item() * len(yb)
        total_correct += (logits.____(1) == yb).sum().item()   # 점수가 가장 높은 선택지 번호
        total_count += len(yb)
    return total_loss / total_count, total_correct / total_count

print("run_epoch 정의 완료")

## 문제 7: 모델 학습

다음을 수행하세요:
1. criterion을 `CrossEntropyLoss()`로 정의하세요.
2. optimizer를 `Adam(model.parameters(), lr=0.01)`로 정의하세요.
3. 50 epoch 동안 학습하세요.
4. 매 epoch마다 train과 validation loss, accuracy를 저장하세요.
5. 매 10 epoch마다 결과를 출력하세요.

**힌트:** `run_epoch` 함수를 사용

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
torch.manual_seed(SEED)
criterion = nn.____()   # 다중분류용 채점 함수
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

EPOCHS = 50
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, ____)   # 학습: optimizer 전달
    va_loss, va_acc = run_epoch(model, ____, criterion)                 # 검증: 어느 loader?
    for key, value in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[key].append(value)
    if epoch % 10 == 0:
        print(f"{epoch:02d} | train {tr_loss:.3f}/{tr_acc:.3f} | val {va_loss:.3f}/{va_acc:.3f}")

## 문제 8: 학습 곡선 시각화

다음을 수행하세요:
1. train loss, validation loss를 한 그래프에 그으세요.
2. train accuracy, validation accuracy를 다른 그래프에 그으세요.
3. 과적합 여부를 판단하세요. (validation loss가 올라가는가?)
4. 최종 validation accuracy를 기록하세요.

**예상:** validation accuracy > 0.9

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["____"], label="validation")     # validation의 loss 기록
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.2)
axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["____"], label="validation")     # validation의 정확도 기록
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()

print("최종 validation accuracy:", round(history["val_acc"][-1], 4))
# 과적합 판단: validation loss가 도중에 되올라가나요? 한 문장으로 적어보세요.

## 문제 9: test 평가

다음을 수행하세요:
1. test loader에서 `run_epoch`를 호출하여 test loss와 accuracy를 계산하세요.
2. test accuracy를 출력하세요.
3. train/val/test 3개 accuracy를 비교하세요.

**참고:** test는 모델 선택이 끝난 후 한 번만 확인합니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
test_loss, test_acc = run_epoch(model, ____, ____)   # 마지막에 딱 한 번 쓰는 loader / 채점 함수
print(f"test loss={test_loss:.3f}, accuracy={test_acc:.3f}")
print("train:", round(history["train_acc"][-1], 3),
      "| val:", round(history["val_acc"][-1], 3),
      "| test:", round(test_acc, 3))

## 문제 10: 예측 확률과 오분류 확인

다음을 수행하세요:
1. model.eval()로 평가 모드로 전환하세요.
2. test 데이터에 대해 softmax 확률을 계산하세요.
3. 예측값과 실제값을 비교하는 DataFrame을 만드세요:
   - 열: true, pred, confidence (최대 확률)
4. confidence가 낮은 샘플 10개를 출력하세요.
5. 오분류된 샘플 개수를 출력하세요.

**실무:** 정확도 하나보다 "어느 샘플이 의심스러운가"를 아는 것이 중요합니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
model.____()          # 평가 모드로 전환
with torch.no_grad():
    X_test_tensor = torch.from_numpy(X_test).to(DEVICE)
    probabilities = model(X_test_tensor).softmax(dim=____).cpu().numpy()   # 선택지 방향으로 합=1

predictions = probabilities.____(axis=1)   # 확률이 가장 높은 선택지 번호
review = pd.DataFrame({
    "true": [class_names[i] for i in y_test],
    "pred": [class_names[i] for i in predictions],
    "confidence": probabilities.max(axis=1).round(3),
})
display(review.sort_values("confidence").head(10))   # 확신 낮은 순 10건
print("misclassified:", int((predictions != y_test).sum()), "/", len(y_test))